In [275]:
import os
from glob import glob
import pandas as pd
from functools import reduce
from xml.etree import ElementTree as et

In [276]:
# Load all xl files and store in a list
xml_list = glob('./data_images/*.xml')

# data cleaning . replace \\ with /
xml_list = list(map(lambda x: x.replace('\\','/'),xml_list))

In [277]:
pip install pandas


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [278]:
xml_list

['./data_images/000001.xml',
 './data_images/000002.xml',
 './data_images/000003.xml',
 './data_images/000004.xml',
 './data_images/000006.xml',
 './data_images/000008.xml',
 './data_images/000010.xml',
 './data_images/000011.xml',
 './data_images/000013.xml',
 './data_images/000014.xml',
 './data_images/000015.xml',
 './data_images/000018.xml',
 './data_images/000022.xml',
 './data_images/000025.xml',
 './data_images/000027.xml',
 './data_images/000028.xml',
 './data_images/000029.xml',
 './data_images/000031.xml',
 './data_images/000037.xml',
 './data_images/000038.xml',
 './data_images/000040.xml',
 './data_images/000043.xml',
 './data_images/000045.xml',
 './data_images/000049.xml',
 './data_images/000053.xml',
 './data_images/000054.xml',
 './data_images/000055.xml',
 './data_images/000056.xml',
 './data_images/000057.xml',
 './data_images/000058.xml',
 './data_images/000059.xml',
 './data_images/000062.xml',
 './data_images/000067.xml',
 './data_images/000068.xml',
 './data_image

In [279]:
# step-2: read xml files
# from each xml file we need to extract
# filename, size(width, height), object(name, xmin, xmax, ymin, ymax)
def extract_text(filename):
    tree = et.parse(filename)
    root =tree.getroot()

    # extract filename
    image_name = root.find('filename').text
    # Width and height of the image
    width = root.find('size').find('width').text
    height = root.find('size').find('height').text

    objs = root.findall('object')
    parser = []
    for obj in objs:
        name=obj.find('name').text
        bndbox = obj.find('bndbox')
        xmin = bndbox.find('xmin').text
        xmax = bndbox.find('xmax').text
        ymin = bndbox.find('ymin').text
        ymax = bndbox.find('ymax').text
        parser.append([image_name, width, height, name, xmin, xmax, ymin, ymax])

    return parser

In [280]:
parser_all = list(map(extract_text, xml_list))
parser_all

[[['000001.jpg', '353', '500', 'dog', '48', '195', '240', '371'],
  ['000001.jpg', '353', '500', 'person', '8', '352', '12', '498']],
 [['000002.jpg', '335', '500', 'train', '139', '207', '200', '301']],
 [['000003.jpg', '500', '375', 'sofa', '123', '215', '155', '195'],
  ['000003.jpg', '500', '375', 'chair', '239', '307', '156', '205']],
 [['000004.jpg', '500', '406', 'car', '13', '84', '311', '362'],
  ['000004.jpg', '500', '406', 'car', '362', '500', '330', '389'],
  ['000004.jpg', '500', '406', 'car', '235', '334', '328', '375'],
  ['000004.jpg', '500', '406', 'car', '175', '252', '327', '364'],
  ['000004.jpg', '500', '406', 'car', '139', '189', '320', '359'],
  ['000004.jpg', '500', '406', 'car', '108', '150', '325', '353'],
  ['000004.jpg', '500', '406', 'car', '84', '121', '323', '350']],
 [['000006.jpg', '500', '375', 'pottedplant', '187', '282', '135', '242'],
  ['000006.jpg', '500', '375', 'diningtable', '154', '369', '209', '375'],
  ['000006.jpg', '500', '375', 'chair', '

In [281]:
data = reduce(lambda x, y : x+y, parser_all)
df  = pd.DataFrame(data, columns = ['filename', 'width', 'height', 'name', 'xmin', 'xmax', 'ymin', 'ymax'])
                                   

In [282]:
df.head()

,filename,width,height,name,xmin,xmax,ymin,ymax
0,000001.jpg,353,500,dog,48,195,240,371
1,000001.jpg,353,500,person,8,352,12,498
2,000002.jpg,335,500,train,139,207,200,301
3,000003.jpg,500,375,sofa,123,215,155,195
4,000003.jpg,500,375,chair,239,307,156,205


In [283]:
df.shape

(14976, 8)

In [284]:
df['name'].value_counts()

name
person         5227
car            1541
chair          1374
bottle          657
pottedplant     592
bird            576
dog             530
sofa            396
horse           395
boat            393
bicycle         389
cat             370
motorbike       369
tvmonitor       361
cow             329
sheep           311
aeroplane       311
train           302
diningtable     299
bus             254
Name: count, dtype: int64

In [285]:
df['name']

0                dog
1             person
2              train
3               sofa
4              chair
            ...     
14971         person
14972         person
14973         person
14974    diningtable
14975            car
Name: name, Length: 14976, dtype: str

In [286]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 14976 entries, 0 to 14975
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   filename  14976 non-null  str  
 1   width     14976 non-null  str  
 2   height    14976 non-null  str  
 3   name      14976 non-null  str  
 4   xmin      14976 non-null  str  
 5   xmax      14976 non-null  str  
 6   ymin      14976 non-null  str  
 7   ymax      14976 non-null  str  
dtypes: str(8)
memory usage: 936.1 KB


In [287]:
# type converison
cols = ['width','height','xmin','xmax', 'ymin','ymax']
df[cols] = df[cols].astype(int)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 14976 entries, 0 to 14975
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   filename  14976 non-null  str  
 1   width     14976 non-null  int64
 2   height    14976 non-null  int64
 3   name      14976 non-null  str  
 4   xmin      14976 non-null  int64
 5   xmax      14976 non-null  int64
 6   ymin      14976 non-null  int64
 7   ymax      14976 non-null  int64
dtypes: int64(6), str(2)
memory usage: 936.1 KB


In [288]:
# Center x, Center y
df['center_x'] =((df['xmax'] + df['xmin'])/2)/df['width']
df['center_y'] =((df['ymax'] + df['ymin'])/2)/df['height']

# w
df['w'] = (df['xmax'] - df['xmin'])/df['width']

#h
df['h'] = (df['ymax'] - df['ymin'])/df['height']

In [289]:
df.head()

,filename,width,height,name,xmin,xmax,ymin,ymax,center_x,center_y,w,h
0,000001.jpg,353,500,dog,48,195,240,371,0.344193,0.611000,0.416431,0.262000
1,000001.jpg,353,500,person,8,352,12,498,0.509915,0.510000,0.974504,0.972000
2,000002.jpg,335,500,train,139,207,200,301,0.516418,0.501000,0.202985,0.202000
3,000003.jpg,500,375,sofa,123,215,155,195,0.338000,0.466667,0.184000,0.106667
4,000003.jpg,500,375,chair,239,307,156,205,0.546000,0.481333,0.136000,0.130667


### Split data into train and test





In [290]:
images = df['filename'].unique()

In [291]:
len(images)

4952

In [292]:
# 80% train and 20% test
img_df = pd.DataFrame(images, columns=['filename'])
img_train = tuple(img_df.sample(frac=0.8)['filename']) # shuffle and pick 80% of images

In [293]:
img_test = tuple(img_df.query(f'filename not in {img_train}')['filename']) # take rest 20% images

In [294]:
len(img_test)

990

In [295]:
len(img_train)

3962

In [296]:
train_df = df.query(f'filename in {img_train}')
test_df = df.query(f'filename in {img_test}')

In [297]:
train_df.head()

,filename,width,height,name,xmin,xmax,ymin,ymax,center_x,center_y,w,h
2,000002.jpg,335,500,train,139,207,200,301,0.516418,0.501000,0.202985,0.202000
3,000003.jpg,500,375,sofa,123,215,155,195,0.338000,0.466667,0.184000,0.106667
4,000003.jpg,500,375,chair,239,307,156,205,0.546000,0.481333,0.136000,0.130667
5,000004.jpg,500,406,car,13,84,311,362,0.097000,0.828818,0.142000,0.125616
6,000004.jpg,500,406,car,362,500,330,389,0.862000,0.885468,0.276000,0.145320


In [298]:
test_df.head()

,filename,width,height,name,xmin,xmax,ymin,ymax,center_x,center_y,w,h
0,000001.jpg,353,500,dog,48,195,240,371,0.344193,0.611000,0.416431,0.262000
1,000001.jpg,353,500,person,8,352,12,498,0.509915,0.510000,0.974504,0.972000
24,000013.jpg,500,375,cow,299,446,160,252,0.745000,0.549333,0.294000,0.245333
50,000031.jpg,500,335,train,41,430,77,255,0.471000,0.495522,0.778000,0.531343
52,000038.jpg,500,375,bicycle,159,279,178,246,0.438000,0.565333,0.240000,0.181333


## Assign id number to object names

In [299]:
# label encoding
def label_encoding(x):
    labels = {'person':0 ,'car':1, 'chair':2 , 'bottle':3, 'pottedplant':4, 'bird':5, 'dog':6, 'sofa':7, 'horse':8,
              'boat':9, 'bicycle':10, 'cat':11, 'motorbike':12, 'tvmonitor':13, 'cow':14, 'sheep':15, 'aeroplane':16, 
              'train':17, 'diningtable':18, 'bus':19}
    return labels[x]

In [300]:
train_df['id'] = train_df['name'].apply(label_encoding)
test_df['id'] = test_df['name'].apply(label_encoding)

In [301]:
train_df.head()

,filename,width,height,name,xmin,xmax,ymin,ymax,center_x,center_y,w,h,id
2,000002.jpg,335,500,train,139,207,200,301,0.516418,0.501000,0.202985,0.202000,17
3,000003.jpg,500,375,sofa,123,215,155,195,0.338000,0.466667,0.184000,0.106667,7
4,000003.jpg,500,375,chair,239,307,156,205,0.546000,0.481333,0.136000,0.130667,2
5,000004.jpg,500,406,car,13,84,311,362,0.097000,0.828818,0.142000,0.125616,1
6,000004.jpg,500,406,car,362,500,330,389,0.862000,0.885468,0.276000,0.145320,1


## Save Image and Labels in text

In [302]:
import os
from shutil import move

In [303]:
train_folder = 'data_images/train'
test_folder = 'data_images/test'

os.mkdir(train_folder)
os.mkdir(test_folder)

In [304]:
cols = ['filename','id', 'center_x', 'center_y', 'w', 'h']
groupby_obj_train = train_df[cols].groupby('filename')
groupby_obj_test = test_df[cols].groupby('filename')

In [305]:
# save each image in train/test folder and respective labels in .txt
def save_data(filename, folder_path, group_obj):
    # move image
    src = os.path.join('data_images', filename)
    dst = os.path.join(folder_path,filename)
    move(src, dst) # move image to the destination folder

    # save the labels
    text_filename = os.path.join(folder_path,
                                 os.path.splitext(filename)[0]+'.txt')
    group_obj.get_group(filename).set_index('filename').to_csv(text_filename, sep=' ',index=False, header=False)
    


In [306]:
groupby_obj_train.groups

{'000002.jpg': [2], '000003.jpg': [3, 4], '000004.jpg': [5, 6, 7, 8, 9, 10, 11], '000006.jpg': [12, 13, 14, 15, 16, 17, 18, 19], '000008.jpg': [20], '000010.jpg': [21, 22], '000011.jpg': [23], '000014.jpg': [25, 26, 27, 28, 29, 30, 31], '000015.jpg': [32], '000018.jpg': [33], '000022.jpg': [34, 35], '000025.jpg': [36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46], '000027.jpg': [47], '000028.jpg': [48], '000029.jpg': [49], '000037.jpg': [51], '000043.jpg': [56, 57, 58], '000045.jpg': [59], '000049.jpg': [60], '000053.jpg': [61], '000054.jpg': [62], '000055.jpg': [63, 64, 65, 66], '000056.jpg': [67], '000057.jpg': [68], '000059.jpg': [73, 74, 75, 76, 77, 78], '000062.jpg': [79], '000067.jpg': [80], '000068.jpg': [81], '000069.jpg': [82, 83], '000070.jpg': [84, 85], '000071.jpg': [86], '000075.jpg': [88], '000076.jpg': [89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99], '000079.jpg': [100], '000080.jpg': [101, 102, 103, 104, 105, 106, 107, 108, 109], '000082.jpg': [110, 111], '000084.jpg': [112, 113

In [307]:
filename_series = pd.Series(groupby_obj_train.groups.keys())
filename_series

0       000002.jpg
1       000003.jpg
2       000004.jpg
3       000006.jpg
4       000008.jpg
           ...    
3957    009953.jpg
3958    009957.jpg
3959    009960.jpg
3960    009962.jpg
3961    009963.jpg
Length: 3962, dtype: str

In [308]:
filename_series.apply(save_data, args=(train_folder,groupby_obj_train))

0       None
1       None
2       None
3       None
4       None
        ... 
3957    None
3958    None
3959    None
3960    None
3961    None
Length: 3962, dtype: object

In [309]:
filename_series_test = pd.Series(groupby_obj_test.groups.keys())
filename_series_test.apply(save_data, args=(test_folder,groupby_obj_test))

0      None
1      None
2      None
3      None
4      None
       ... 
985    None
986    None
987    None
988    None
989    None
Length: 990, dtype: object